<a href="https://colab.research.google.com/github/gabrieelsky/rps-cnn/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
! git clone https://github.com/gabrieelsky/rps-cnn
! mv rps-cnn/* .
! rm -r rps-cnn/ sample_data/

! mkdir -p data/raw
! curl -L -o data/raw/rockpaperscissors.zip https://www.kaggle.com/api/v1/datasets/download/drgfreeman/rockpaperscissors
! unzip -q data/raw/rockpaperscissors.zip -d data/raw/
! rm data/raw/rockpaperscissors.zip
! mkdir saved_models

Cloning into 'rps-cnn'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 56 (delta 20), reused 41 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 281.25 KiB | 1.35 MiB/s, done.
Resolving deltas: 100% (20/20), done.
mv: cannot move 'rps-cnn/notebooks' to './notebooks': Directory not empty
mv: cannot move 'rps-cnn/src' to './src': Directory not empty
rm: cannot remove 'sample_data/': No such file or directory
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  305M  100  305M    0     0   128M      0  0:00:02  0:00:02 --:--:--  176M
replace data/raw/README_rpc-cv-images.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from src.data_loader import create_dataloaders, get_class_mapping
from src.models import BaselineCNN
from src.train import train_model, run_grid_search
from src.evaluate import evaluate_model
from src.config import *

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = get_device()
print(f"Hardware configuration: Using device '{device}'\n")

class_mapping = get_class_mapping(RAW_DATA_DIR)
num_classes = len(class_mapping)
print(f"{num_classes} classes found: {class_mapping}\n")

Hardware configuration: Using device 'mps'

3 classes found: {'paper': 0, 'rock': 1, 'scissors': 2}



In [2]:
# Define hyperparameter grid for Stratified K-Fold CV using run_grid_search
param_grid = {
    'lr': [1e-2, 1e-3, 3e-4],
    'batch_size': [16, 32],
    'dropout_rate': [0.3, 0.5],
    'weight_decay': [0, 1e-4],
    'epochs': [5] # To be set as 20 for the real tuning
}

print('Initiating Stratified K-Fold Grid Search (this may take a while)...')
tuning_results = run_grid_search(BaselineCNN, param_grid, None, RAW_DATA_DIR, device=device, num_classes=num_classes, n_splits=3, seed=RANDOM_SEED, patience=3, min_delta=1e-4)
tuning_results.to_csv('saved_models/grid_search_results.csv', index=False)
print('Grid search complete. Top results:')
print(tuning_results.head())

Initiating Stratified K-Fold Grid Search (this may take a while)...
Starting Grid Search with 24 configurations (Stratified 3-Fold CV)...


Experiment 1/24: {'lr': 0.01, 'batch_size': 16, 'dropout_rate': 0.3, 'weight_decay': 0, 'epochs': 5}
  Fold 1/3 (train 1458 / val 730)

--- Epoch 1/5 ---


NameError: name 'AverageMeter' is not defined

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [2]:
# Extract best configuration from tuning results
#best_config = tuning_results.iloc[0]
best_config = {
    'lr': 1e-3,
    'batch_size': 16,
    'dropout_rate': 0.5,
    'weight_decay': 1e-4,
    'epochs': 20
}

best_lr = float(best_config['lr'])
best_batch_size = int(best_config['batch_size'])
best_dropout = float(best_config.get('dropout_rate', 0.0))
best_weight_decay = float(best_config.get('weight_decay', 0.0))
best_epochs = int(best_config.get('epochs', 5))
final_epochs = 20  # Train longer for the final run
print(f"Selected best config: lr={best_lr}, batch_size={best_batch_size}, dropout={best_dropout}, weight_decay={best_weight_decay}, epochs={best_epochs}")

Selected best config: lr=0.001, batch_size=16, dropout=0.5, weight_decay=0.0001, epochs=20


In [3]:
print(f"\nStarting final training with best parameters: LR={best_lr}, Batch={best_batch_size}, Dropout={best_dropout}")

train_loader, val_loader, test_loader, _ = create_dataloaders(RAW_DATA_DIR, batch_size=best_batch_size)

model = BaselineCNN(num_classes=num_classes, input_shape=(3, IMG_HEIGHT, IMG_WIDTH)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)

model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=final_epochs,
    patience=5,
    min_delta=1e-4
)

print("\nEvaluating on the test set...")
test_loss, all_preds, all_labels = evaluate_model(
    model=model,
    test_loader=test_loader,
    criterion=criterion,
    device=device,
    class_mapping=class_mapping,
)

save_path = os.path.join(MODELS_DIR, "convnet_best.pth")
torch.save(model.state_dict(), save_path)
print(f"\nModel weights successfully saved to {save_path}")


Starting final training with best parameters: LR=0.001, Batch=16, Dropout=0.5

--- Epoch 1/20 ---
[TRAIN Batch 000/096]	Time 0.456s (0.456s)	Loss 1.1040 (1.1040)	Prec@1  25.0 ( 25.0)
[TRAIN Batch 010/096]	Time 0.081s (0.120s)	Loss 1.0863 (1.0989)	Prec@1  50.0 ( 32.4)
[TRAIN Batch 020/096]	Time 0.083s (0.103s)	Loss 1.0996 (1.0996)	Prec@1  31.2 ( 32.4)
[TRAIN Batch 030/096]	Time 0.084s (0.098s)	Loss 1.0997 (1.0993)	Prec@1  31.2 ( 32.9)
[TRAIN Batch 040/096]	Time 0.076s (0.096s)	Loss 1.0863 (1.0976)	Prec@1  37.5 ( 33.2)
[TRAIN Batch 050/096]	Time 0.080s (0.094s)	Loss 1.0775 (1.0970)	Prec@1  43.8 ( 33.3)
[TRAIN Batch 060/096]	Time 0.082s (0.092s)	Loss 1.1004 (1.0968)	Prec@1  18.8 ( 32.6)
[TRAIN Batch 070/096]	Time 0.080s (0.092s)	Loss 1.1046 (1.0965)	Prec@1  31.2 ( 33.9)
[TRAIN Batch 080/096]	Time 0.082s (0.090s)	Loss 1.0853 (1.0956)	Prec@1  43.8 ( 35.5)
[TRAIN Batch 090/096]	Time 0.083s (0.089s)	Loss 1.0790 (1.0952)	Prec@1  62.5 ( 35.6)

===============> Total time 8s	Avg loss 1.0952	Avg

<Figure size 640x480 with 0 Axes>

KeyboardInterrupt: 